# Day 2, Session 1 -- Demos: OpenAI API Engineering with Structured Outputs

**Engineering context:** You are the engineer building production-grade API integrations. Consultants are your users -- they need reliable structured outputs, streaming for real-time briefings, and validated data pipelines they can trust.

## Session Goal

By the end of this session you will master **production-grade API patterns** -- streaming, token tracking, JSON mode, function calling, Pydantic validation, and extraction pipelines. These are the engineering foundations for building reliable LLM applications.

| Demo | Pattern | Why It Matters |
|------|---------|----------------|
| 1 | Streaming + Token Tracking | Real-time output delivery; cost management across engagements |
| 2 | JSON Mode | Guaranteed valid JSON for downstream systems |
| 3 | Function Calling | Model-driven tool selection; sandboxed execution |
| 4 | Pydantic Validation | Type-safe parsing; catch schema mismatches before they propagate |
| 5 | Extraction Pipeline | Reusable pattern combining JSON mode + Pydantic for unstructured data |

In [ ]:
!pip install -q openai pydantic python-dotenv

## Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import openai
import json
import os
from pydantic import BaseModel, Field
from typing import Optional, List

print("All imports successful!")
print(f"OpenAI SDK version: {openai.__version__}")

---
## Demo 1: OpenAI API Deep Dive -- Streaming, Token Usage, and Finish Reasons

In production systems you need to: (a) track token usage for cost management, (b) stream responses for real-time briefings, and (c) inspect finish reasons to know if the model completed its analysis or was cut off.

**Scenario:** A partner asks the AI assistant to generate a concise market overview for a CEO briefing. We track tokens to manage costs across engagements and stream the response so the delivery team can review output in real time.

### Demo 1a: Token Usage Tracking

Every API call returns a `usage` object showing how many tokens were consumed. This is essential for:
- Budgeting costs per engagement
- Alerting when prompts are unexpectedly large
- Comparing cost across models

In [ ]:
# Demo 1a - Token Usage Tracking

client = openai.OpenAI()

response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "Provide a 3-sentence executive summary of the global healthcare market outlook for a CEO briefing."}],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "150"))
)

print("=== Token Usage (Cost Tracking) ===")
print(f"Prompt tokens    : {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens     : {response.usage.total_tokens}")
print(f"Finish reason    : {response.choices[0].finish_reason}")
print(f"\nCEO Briefing Response:\n{response.choices[0].message.content}")

**Observe:** The `finish_reason` tells you whether the model completed naturally (`stop`) or ran out of token budget (`length`). In production, always check this -- a `length` finish means the response was truncated and may be incomplete.

### Demo 1b: Streaming Responses

Streaming delivers tokens one at a time as they are generated. This is critical for:
- Real-time briefing displays where consultants start reading immediately
- Long-running generations where you want progress feedback
- Chat interfaces that feel responsive

In [ ]:
# Demo 1b - Streaming Responses

client = openai.OpenAI()

print("=== Streaming Response (Real-Time Review) ===")
stream = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "List the top 3 priorities for a digital transformation engagement at a Fortune 500 retailer, one per line."}],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "100")),
    stream=True
)

collected_text = ""
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        token = chunk.choices[0].delta.content
        collected_text += token
        print(token, end="", flush=True)

print(f"\n\nFull collected text length: {len(collected_text)} characters")

**Observe:** Tokens appear one at a time, mimicking a 'typing' effect. In production, this lets consultants start reading immediately instead of waiting for the full response. Notice that with streaming, you do not get token usage statistics in each chunk -- you must count them yourself or use the final chunk's usage field (available in newer SDK versions with `stream_options={"include_usage": True}`).

### Try This

Set `max_tokens=20` in the streaming call. What happens to the output? Check the last chunk's `finish_reason` -- is it `stop` or `length`?

---
## Demo 2: Structured Output with JSON Mode -- Client Profile Extraction

By setting `response_format={"type": "json_object"}`, the model is guaranteed to return valid JSON. This is critical for workflows where downstream systems must parse data reliably. You **must** include the word "JSON" in your prompt when using this mode.

**Scenario:** An engagement manager needs a structured client profile for a new healthcare engagement. JSON mode ensures the profile data is machine-readable for CRM and engagement tracking systems.

In [ ]:
# Demo 2 - Structured Output with JSON Mode

client = openai.OpenAI()

response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": "You are an engagement assistant that outputs JSON."},
        {"role": "user", "content": "Create a client profile for a new engagement with Acme Healthcare Corp. Include: client_name, industry, annual_revenue_usd, number_of_employees, headquarters, key_challenges (list), and engagement_type."}
    ],
    response_format={"type": "json_object"},
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
)

raw_json = response.choices[0].message.content
print("Raw JSON response:")
print(raw_json)

# Parse and pretty-print
parsed = json.loads(raw_json)
print("\nParsed and formatted:")
print(json.dumps(parsed, indent=2))

# Access individual fields
print(f"\nClient      : {parsed.get('client_name', 'N/A')}")
print(f"Industry    : {parsed.get('industry', 'N/A')}")
print(f"Engagement  : {parsed.get('engagement_type', 'N/A')}")

**Gotcha:** You MUST include the word 'JSON' in your prompt when using `response_format={"type": "json_object"}`. If you forget, the API will return an error. In this demo, the system message says "outputs JSON" -- that satisfies the requirement. Also note that JSON mode guarantees *syntactically valid* JSON, but says nothing about which keys or types are present. That is where Pydantic (Demo 4) comes in.

---
## Demo 3: Function Calling -- Consulting Research Tools

Function calling lets the model decide **when** and **how** to call external tools. This enables an AI assistant to autonomously pull market data, run benchmarks, or look up competitor intelligence. The flow is:
1. You define tool schemas (e.g., `market_research`) and send them with the request
2. The model returns a `tool_calls` response (instead of regular content)
3. You execute the function locally (simulated data)
4. You send the result back to the model for a final answer

**Scenario:** An associate asks the AI for market research on a specific industry. The model decides to call the `market_research` tool to retrieve structured market data.

### Demo 3a: Define Tools and Simulated Function

First, we define the tool schema (what the model can call) and the actual function implementation (what runs on our side).

In [ ]:
# Demo 3a - Define Tools and Simulated Function

client = openai.OpenAI()

# Tool schema: tells the model what functions are available
tools = [
    {
        "type": "function",
        "function": {
            "name": "market_research",
            "description": "Get market research data for a specific industry sector",
            "parameters": {
                "type": "object",
                "properties": {
                    "industry": {
                        "type": "string",
                        "description": "The industry sector, e.g., 'healthcare', 'financial_services', 'energy'"
                    },
                    "region": {
                        "type": "string",
                        "enum": ["north_america", "europe", "asia_pacific", "global"],
                        "description": "Geographic region for the analysis"
                    }
                },
                "required": ["industry"]
            }
        }
    }
]

# Simulated function: in production, this would call a real API or database
def market_research(industry, region="global"):
    """Simulated market research function returning industry data."""
    market_data = {
        "healthcare": {"market_size_usd_bn": 8450, "cagr_pct": 7.9, "key_trend": "AI-driven diagnostics and personalized medicine"},
        "financial_services": {"market_size_usd_bn": 28500, "cagr_pct": 6.2, "key_trend": "Embedded finance and real-time payments"},
        "energy": {"market_size_usd_bn": 6700, "cagr_pct": 8.5, "key_trend": "Energy transition and grid modernization"},
    }
    data = market_data.get(industry, {"market_size_usd_bn": 5000, "cagr_pct": 5.0, "key_trend": "Digital transformation"})
    return json.dumps({"industry": industry, "region": region, "market_size_usd_bn": data["market_size_usd_bn"], "cagr_pct": data["cagr_pct"], "key_trend": data["key_trend"]})

print("Tools defined:", [t["function"]["name"] for t in tools])
print("\nExample function call:")
print(market_research("healthcare", "global"))

**Observe:** The tool schema is a JSON Schema that describes the function's name, description, and parameters. The model uses this schema to decide (a) whether to call the function and (b) what arguments to pass. The actual function implementation is completely separate -- the model never sees or executes it.

### Demo 3b: Make the Initial API Call

Now we send a user question along with the tools list. The model will decide whether to answer directly or request a function call.

In [ ]:
# Demo 3b - Make the Initial API Call

response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "What does the healthcare market look like globally? I need data for a client engagement."}],
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message
print(f"Finish reason: {response.choices[0].finish_reason}")
print(f"Has tool_calls: {message.tool_calls is not None}")

if message.tool_calls:
    tool_call = message.tool_calls[0]
    print(f"\nFunction the model wants to call: {tool_call.function.name}")
    print(f"Arguments it chose: {tool_call.function.arguments}")
else:
    print(f"Direct response (no tool call): {message.content}")

**Observe:** When the model decides to call a function, the `finish_reason` is `tool_calls` (not `stop`). The model has NOT executed anything -- it has simply told us which function to call and what arguments to use. This is purely a *request*.

### Demo 3c: Execute the Function and Send the Result Back

Now we complete the loop: execute the function with the model's chosen arguments, then send the result back so the model can formulate a final answer.

In [ ]:
# Demo 3c - Execute Function and Send Result Back

if message.tool_calls:
    tool_call = message.tool_calls[0]
    
    # Execute the function with the model's chosen arguments
    args = json.loads(tool_call.function.arguments)
    result = market_research(**args)
    print(f"Function executed with args: {args}")
    print(f"Function result: {result}")
    
    # Send the result back to the model for a final answer
    follow_up = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "user", "content": "What does the healthcare market look like globally? I need data for a client engagement."},
            message,  # The assistant's message with tool_calls
            {"role": "tool", "tool_call_id": tool_call.id, "content": result}
        ],
        tools=tools
    )
    print(f"\n=== Final Response ===\n{follow_up.choices[0].message.content}")
else:
    print("No tool call was made -- model answered directly.")

**Key Insight:** The model does not execute functions -- it tells YOU which function to call and with what arguments. Your code runs the function, then sends the result back. This separation keeps the model sandboxed. The complete loop is:

```
User Question --> Model decides to call tool --> YOUR CODE executes function --> Result sent back --> Model writes final answer
```

This pattern is the foundation for all agentic systems.

### Try This

Add a second tool definition (e.g., `competitor_analysis`) to the tools list. Ask a question that requires it (e.g., "Who are the top competitors in the energy sector?"). Does the model choose the correct tool?

### Quick Recap

- **Q:** When would you use JSON mode (Demo 2) vs function calling (Demo 3)? What is the key difference?
- **Q:** What happens if the model returns a function call for a tool you have not implemented?

---
## Demo 4: Pydantic-Based Response Validation -- Engagement Summary

Pydantic models let you define the **exact shape** of the data you expect from the LLM, with type checking and constraints. If the LLM returns data that does not match, Pydantic will raise an error -- catching problems before they propagate through your workflow.

**Scenario:** Validate a structured engagement summary for a recently completed strategy engagement. The Pydantic model enforces that all required fields (client name, industry, engagement type, satisfaction score, etc.) are present and correctly typed.

### Demo 4a: Define the Pydantic Model

We define a strict schema using Pydantic. Note the use of `Field` with constraints like `ge=1.0, le=5.0` -- these enforce business rules at the validation layer.

In [ ]:
# Demo 4a - Define Pydantic Model for Engagement Summary

class EngagementSummary(BaseModel):
    client_name: str = Field(description="Name of the client organization")
    industry: str = Field(description="Client industry sector")
    engagement_type: str = Field(description="Type of engagement, e.g., Strategy, M&A Due Diligence, Digital Transformation")
    duration_weeks: int = Field(description="Duration of the engagement in weeks")
    satisfaction_score: float = Field(ge=1.0, le=5.0, description="Client satisfaction score (1-5)")
    key_recommendation: str = Field(description="One-sentence primary recommendation")
    follow_on_opportunity: bool = Field(description="Whether there is a follow-on engagement opportunity")

print("Pydantic model schema:")
print(json.dumps(EngagementSummary.model_json_schema(), indent=2))

**Observe:** The Pydantic model generates a JSON Schema automatically. This schema could even be passed to the LLM as part of the prompt to guide its output. The key benefit: if the LLM returns a `satisfaction_score` of 7.0 (out of range) or forgets `client_name`, Pydantic will raise a `ValidationError` immediately.

### Demo 4b: Request Structured Output and Validate

Now we combine JSON mode (guaranteed valid JSON) with Pydantic validation (guaranteed correct schema).

In [ ]:
# Demo 4b - Request and Validate with Pydantic

client = openai.OpenAI()

response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": (
            "You are an engagement tracking system. Output your engagement summary as JSON with these exact fields: "
            "client_name (string), industry (string), engagement_type (string), duration_weeks (integer), "
            "satisfaction_score (float 1-5), key_recommendation (string), follow_on_opportunity (boolean)."
        )},
        {"role": "user", "content": "Summarize the recently completed digital transformation engagement with Meridian Financial Group."}
    ],
    response_format={"type": "json_object"},
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
)

raw = response.choices[0].message.content
print("Raw JSON from LLM:")
print(raw)

# Validate with Pydantic
try:
    summary = EngagementSummary.model_validate_json(raw)
    print("\nValidation PASSED")
    print(f"  Client          : {summary.client_name}")
    print(f"  Industry        : {summary.industry}")
    print(f"  Engagement Type : {summary.engagement_type}")
    print(f"  Duration        : {summary.duration_weeks} weeks")
    print(f"  Satisfaction    : {summary.satisfaction_score}/5.0")
    print(f"  Recommendation  : {summary.key_recommendation}")
    print(f"  Follow-on       : {'Yes' if summary.follow_on_opportunity else 'No'}")
except Exception as e:
    print(f"\nValidation FAILED: {e}")

**Gotcha:** `response_format={"type": "json_object"}` guarantees valid JSON but does NOT guarantee the schema matches your Pydantic model. The LLM might return `{"name": "..."}` instead of `{"client_name": "..."}`, or omit a required field entirely. Pydantic catches those mismatches -- that is why you need both layers working together.

In production, when validation fails you have several options:
1. Retry the request with a more explicit prompt
2. Send the error message back to the LLM and ask it to fix the output
3. Use default values for non-critical fields

---
## Demo 5: Building a Structured Data Extraction Pipeline -- Engagement Team Extraction

This demo combines JSON mode and Pydantic into a reusable pipeline that extracts structured team information from unstructured engagement notes. This pattern is the foundation for many agentic data-processing workflows -- from staffing databases to knowledge management systems.

**Scenario:** An engagement kickoff email contains team member details in unstructured text. We build a pipeline to extract each team member's name, role, email, practice area, and office location into structured data.

### Demo 5a: Define the TeamMember Model and Extraction Function

Notice the use of `Optional` fields -- not all information may be present in the source text, and our model handles that gracefully.

In [ ]:
# Demo 5a - Define TeamMember Model and Extraction Function

client = openai.OpenAI()

class TeamMember(BaseModel):
    name: str = Field(description="Full name of the team member")
    email: Optional[str] = Field(default=None, description="Email address if found")
    phone: Optional[str] = Field(default=None, description="Phone number if found")
    role: Optional[str] = Field(default=None, description="Consulting role: Partner, Associate Partner, Engagement Manager, Associate, Business Analyst")
    practice: Optional[str] = Field(default=None, description="Practice area if found")

def extract_team_members(text: str) -> List[TeamMember]:
    """Extract structured team member information from unstructured engagement notes."""
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": (
                "Extract all consulting team member information from the text. "
                "Return a JSON object with a 'team_members' key containing a list. "
                "Each member should have: name, email (null if not found), "
                "phone (null if not found), role (null if not found), "
                "practice (null if not found)."
            )},
            {"role": "user", "content": text}
        ],
        response_format={"type": "json_object"},
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "500"))
    )

    data = json.loads(response.choices[0].message.content)
    members = []
    for item in data.get("team_members", []):
        try:
            members.append(TeamMember(**item))
        except Exception as e:
            print(f"  [SKIP] Invalid team member entry: {e}")
    return members

print("TeamMember model and extract_team_members() function defined.")
print(f"TeamMember fields: {list(TeamMember.model_fields.keys())}")

### Demo 5b: Test the Extraction Pipeline

Let us run the pipeline on a realistic engagement kickoff email.

In [ ]:
# Demo 5b - Test the Extraction Pipeline

sample_text = """
Team, here is the staffing for the Apex Energy transformation engagement:

The engagement will be led by Partner Rajesh Gupta from our Houston office (Energy Practice).
Reach him at rajesh.gupta@mckinsey.com or (713) 555-0142.

Engagement Manager Sarah Okonkwo from the Chicago office will run day-to-day operations.
She is part of our Operations Practice. Her email is sarah.okonkwo@mckinsey.com.

We also have two Associates staffed: David Chen (Digital/McKinsey Digital, New York office,
david.chen@mckinsey.com) and Priya Sharma (Strategy & Corporate Finance, London office).

For analytics support, reach out to Business Analyst Tom Williams at tom.williams@mckinsey.com.
"""

print("Input text:")
print(sample_text)
print("=" * 60)
print("Extracted engagement team:")
print("=" * 60)

members = extract_team_members(sample_text)
for i, member in enumerate(members, 1):
    print(f"\nTeam Member {i}:")
    print(f"  Name     : {member.name}")
    print(f"  Email    : {member.email or 'N/A'}")
    print(f"  Phone    : {member.phone or 'N/A'}")
    print(f"  Role     : {member.role or 'N/A'}")
    print(f"  Practice : {member.practice or 'N/A'}")

print(f"\nTotal members extracted: {len(members)}")

**Observe:** Priya Sharma has no email listed in the source text. The `Optional[str]` field with `default=None` handles this gracefully -- no error is raised, and the field is simply `None`. This is a key design pattern: make fields optional when the source data may be incomplete, and use `Field(default=None)` to set a sensible default.

The `try/except` in the extraction loop means one malformed entry will not crash the entire pipeline -- it skips the bad entry and continues processing the rest.

### Quick Recap

- **Q:** Why is Pydantic validation important even when using JSON mode?
- **Q:** What would happen if the unstructured text mentions a team member without an email -- how does the `Optional` field handle it?
- **Q:** In the extraction pipeline, why do we wrap each `TeamMember(**item)` in a try/except instead of letting the whole function fail?

---
## Summary

In this demo notebook you saw five production-grade OpenAI API patterns:

1. **Streaming and Token Tracking** -- Real-time output delivery and cost management.
2. **JSON Mode** -- Guaranteed valid JSON via `response_format={"type": "json_object"}`.
3. **Function Calling** -- Defining tools the model can invoke, handling the request-execute-respond loop.
4. **Pydantic Validation** -- Type-safe parsing and validation of LLM outputs.
5. **Extraction Pipelines** -- Combining JSON mode and Pydantic into reusable data extraction workflows.

**Key Takeaway:** In production LLM applications, you always need multiple layers of reliability:
- JSON mode ensures syntactically valid output
- Pydantic ensures semantically correct output
- Function calling keeps the model sandboxed while giving it access to real tools
- Streaming and token tracking keep the UX responsive and costs predictable

**Next:** Open `D2S1_exercises.ipynb` to practice building these patterns yourself.